In [1]:
# 收集 inference_results_single下的每个文件夹下的summary.json 中的mae值
import os
import json
import re
import pandas as pd
import numpy as np
import glob

def get_mae_from_json(json_path):
    """
    从json文件中提取mae值
    :param json_path: json文件路径
    :return: mae值
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
        mae = data['metrics']['mae']
    return mae

json_paths=glob.glob(os.path.join('inference_win71_single_results', '**', 'summary.json'), recursive=True)
mae_values = []
for json_path in json_paths:
    mae = get_mae_from_json(json_path)
    # 提取文件夹名称
    folder_name = os.path.basename(os.path.dirname(json_path))
    mae_values.append((folder_name, mae))
# mae_values 保存 csv 文件
df = pd.DataFrame(mae_values, columns=['folder_name', 'mae'])
df['mae'] = df['mae'].astype(float)
# 计算平均值
df['mae'] = df['mae'].replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df = df.groupby([ 'folder_name']).agg({'mae': 'mean'}).reset_index()
# 保存为csv文件
df.to_csv('mae_results.csv', index=False)
